# Cache Dependency Graph — Deep Dive

cachepy automatically tracks **which cached function called which** and **which files each depends on**.  
This notebook explores the full graph API: building, inspecting, persisting, and visualising the dependency DAG.

| # | Section | What it shows |
|---|---------|---------------|
| 1 | [Building a Pipeline](#1.-Building-a-Pipeline) | Nested cached calls create parent/child edges |
| 2 | [Inspecting Nodes](#2.-Inspecting-Nodes) | `cache_tree_nodes()` metadata |
| 3 | [File Tracking](#3.-File-Tracking-with-track_file) | `track_file()` registers file dependencies |
| 4 | [Staleness Detection](#4.-Staleness-Detection) | `cache_tree_changed_files()` finds outdated nodes |
| 5 | [Visualisation](#5.-Visualisation) | `plot_cache_graph()` with colour coding |
| 6 | [Persistence](#6.-Graph-Persistence) | Save/load/sync graph to disk |
| 7 | [Querying by File](#7.-Querying-by-File) | `cache_tree_for_file()` reverse lookup |
| 8 | [Complex DAG](#8.-Complex-DAG) | Diamond dependencies and fan-out |

In [ ]:
import os, sys, time, shutil
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from cachepy import (
    cache_file, cache_tree_nodes, cache_tree_reset,
    cache_tree_save, cache_tree_load, cache_tree_sync,
    cache_tree_changed_files, cache_tree_for_file,
    track_file, plot_cache_graph,
)
from cachepy.cache_file import _file_state_cache

import matplotlib.pyplot as plt

DEMO = Path("graph_demo_cache")

def fresh():
    if DEMO.exists(): shutil.rmtree(DEMO)
    cache_tree_reset()
    _file_state_cache.clear()
    return DEMO

print("Ready.")

---
## 1. Building a Pipeline

When a cached function calls another cached function, cachepy records the **parent → child** relationship automatically.

In [ ]:
cache_dir = fresh()

@cache_file(cache_dir)
def load_counts(path):
    """Step 1: load raw counts from file."""
    return {"genes": ["TP53", "BRCA1", "EGFR"], "counts": [100, 250, 80]}

@cache_file(cache_dir)
def normalize(counts):
    """Step 2: normalize counts."""
    total = sum(counts["counts"])
    return {g: c / total for g, c in zip(counts["genes"], counts["counts"])}

@cache_file(cache_dir)
def top_genes(normalized, n=2):
    """Step 3: pick top N genes."""
    return sorted(normalized.items(), key=lambda x: -x[1])[:n]

@cache_file(cache_dir)
def pipeline(input_path, n=2):
    """Full pipeline: load → normalize → top genes."""
    raw = load_counts(input_path)
    normed = normalize(raw)
    return top_genes(normed, n=n)

result = pipeline("samples/counts.csv")
print(f"Top genes: {result}")

---
## 2. Inspecting Nodes

Each node in the graph stores: function name, hash, parent/child links, tracked files, and the cache file path.

In [ ]:
nodes = cache_tree_nodes()
print(f"Total nodes: {len(nodes)}\n")

for nid, node in nodes.items():
    print(f"Node: {node['fname']}")
    print(f"  ID:       {nid}")
    print(f"  Parents:  {node.get('parents', [])}")
    print(f"  Children: {node.get('children', [])}")
    print(f"  Files:    {node.get('files', [])}")
    print(f"  Outfile:  {node.get('outfile', 'N/A')}")
    print()

---
## 3. File Tracking with `track_file()`

`track_file(path)` registers an explicit file dependency on the **current** graph node.  
The file's content hash is stored so changes can be detected later.

In [ ]:
cache_dir = fresh()

# Create a data file
data_file = Path("graph_demo_data.tsv")
data_file.write_text("gene\tcount\nTP53\t100\nBRCA1\t250\n")

@cache_file(cache_dir)
def read_data(path):
    p = track_file(path)          # <-- register dependency
    lines = p.read_text().strip().split("\n")
    return [l.split("\t") for l in lines[1:]]

@cache_file(cache_dir)
def summarize(path):
    rows = read_data(path)
    return {r[0]: int(r[1]) for r in rows}

print(summarize(str(data_file)))

# Show tracked files per node
for nid, node in cache_tree_nodes().items():
    fh = node.get('file_hashes', {})
    if fh:
        print(f"\n{node['fname']} tracks {len(fh)} file(s):")
        for fp, h in fh.items():
            print(f"  {fp}  hash={h[:16]}...")

---
## 4. Staleness Detection

`cache_tree_changed_files()` compares stored file hashes against current disk state.  
Nodes whose tracked files changed are flagged as **stale**.

In [ ]:
# Before modification — nothing stale
stale = cache_tree_changed_files()
print(f"Stale nodes before edit: {len(stale)}")

# Modify the file
data_file.write_text("gene\tcount\nTP53\t100\nBRCA1\t250\nEGFR\t500\n")
_file_state_cache.clear()

# After modification — read_data should be stale
stale = cache_tree_changed_files()
print(f"Stale nodes after edit: {len(stale)}")
for nid, info in stale.items():
    print(f"  {info['node']['fname']}: {info['changed_files']}")

---
## 5. Visualisation

`plot_cache_graph()` renders the DAG with matplotlib.  
- **Navy** = cached and up-to-date  
- **Amber** = stale (tracked file changed)  
- **Gray** = cache file missing  
- **Light blue** = tracked file node

In [ ]:
fig = plot_cache_graph(highlight_stale=True)
plt.title("Cache Graph — after file modification", fontsize=14, fontweight="bold")
plt.show()

In [ ]:
# Save to file
fig = plot_cache_graph(output="graph_demo.png", highlight_stale=True)
print(f"Saved to graph_demo.png ({Path('graph_demo.png').stat().st_size} bytes)")

---
## 6. Graph Persistence

The graph can be saved and loaded independently of the cache files.  
It's also auto-persisted to `graph.pkl` in the cache directory.

In [ ]:
# Save graph
save_path = cache_dir / "my_graph.pkl"
cache_tree_save(save_path)
print(f"Saved {len(cache_tree_nodes())} nodes to {save_path}")

# Clear and verify empty
cache_tree_reset()
print(f"After reset: {len(cache_tree_nodes())} nodes")

# Load back
cache_tree_load(save_path)
print(f"After load:  {len(cache_tree_nodes())} nodes")

# Sync merges disk graph into memory (non-destructive)
cache_tree_reset()
cache_tree_sync(cache_dir)  # reads graph.pkl
print(f"After sync:  {len(cache_tree_nodes())} nodes")

---
## 7. Querying by File

`cache_tree_for_file(path)` finds all graph nodes that depend on a specific file.

In [ ]:
dependents = cache_tree_for_file(str(data_file.resolve()))
print(f"Nodes depending on {data_file.name}:")
for nid, node in dependents.items():
    print(f"  {node['fname']} ({nid})")

# Query a file that nothing depends on
unrelated = cache_tree_for_file("/nonexistent/file.txt")
print(f"\nNodes depending on nonexistent file: {len(unrelated)}")

---
## 8. Complex DAG

Build a diamond-shaped dependency graph: two branches merge into a final step.

In [ ]:
cache_dir = fresh()

@cache_file(cache_dir)
def fetch_expression(sample):
    return {"TP53": 10, "BRCA1": 25, "EGFR": 8, "sample": sample}

@cache_file(cache_dir)
def fetch_mutations(sample):
    return {"TP53": True, "BRCA1": False, "EGFR": True, "sample": sample}

@cache_file(cache_dir)
def branch_expression(sample):
    expr = fetch_expression(sample)
    return {g: v for g, v in expr.items() if isinstance(v, (int, float)) and v > 9}

@cache_file(cache_dir)
def branch_mutations(sample):
    muts = fetch_mutations(sample)
    return [g for g, v in muts.items() if v is True]

@cache_file(cache_dir)
def integrate(sample):
    """Diamond merge: combine expression + mutation branches."""
    high_expr = branch_expression(sample)
    mutated = branch_mutations(sample)
    return {g: high_expr[g] for g in mutated if g in high_expr}

result = integrate("patient_001")
print(f"Genes with high expression AND mutations: {result}\n")

# Show the graph
print(f"Nodes: {len(cache_tree_nodes())}")
for nid, node in cache_tree_nodes().items():
    nc = len(node.get('children', []))
    np_ = len(node.get('parents', []))
    print(f"  {node['fname']:25s}  children={nc}  parents={np_}")

In [ ]:
fig = plot_cache_graph()
plt.title("Diamond DAG — expression + mutation integration", fontsize=14, fontweight="bold")
plt.show()

---
## Cleanup

In [ ]:
for f in ["graph_demo_data.tsv", "graph_demo.png"]:
    Path(f).unlink(missing_ok=True)
if DEMO.exists(): shutil.rmtree(DEMO)
print("Done.")